# Measuring the agent, for real

Every orchestration number this project has published comes from the **scripted
planner** — a deterministic keyword policy that speaks the tool protocol. It is the
control, not the result. This notebook runs the same S0–S3 ladder with served
open-weight models across a quantization sweep, and scores each cell.

**What it measures**

| metric | what it answers |
|---|---|
| tool-call validity | can the model address the tools at all? |
| unsupported-claim rate | what share of its numeric claims does no tool output support? |
| planning quality | how far did its choices move the answer from the S0 reference? |
| verifier delta | true statements the model verifier removed, against hallucinations it let through |

**What it does *not* measure honestly here: wall-clock.** A Colab GPU is shared and
throttled, you do not control which card you get, and `bitsandbytes` int8 is slower
than fp16 on pre-Ampere cards — so a quantization timing comparison on a T4 measures
the kernel, not the model. Timings are recorded and **flagged**; `timing_warnings()`
at the end tells you when they must not be compared. Take timing separately, on one
fixed machine, or leave it out.

## 1 · Install

About three minutes. `bitsandbytes` is only needed for the 8-bit and 4-bit rows.

In [ ]:
!git clone -q https://github.com/berdakh/onset-hfo.git 2>/dev/null || true
%cd /content/onset-hfo/onset-hfo
!pip install -q -e ".[dev]"
!pip install -q bitsandbytes accelerate

## 2 · What card did you get?

Run this before anything else. It decides which cells are worth attempting and
whether you need to force `float16`.

In [ ]:
import torch
from onset_agent.benchmark import APPROX_VRAM_GB, environment, fits

env = environment()
print(f"GPU          {env['gpu_name']}  ({env['gpu_total_gb']} GB)")
print(f"CUDA         {env['cuda']}")
print(f"bfloat16     {env['supports_bf16']}")

# Most recent model configs declare bfloat16. On a card without it -- a T4 --
# dtype='auto' will honour a dtype the hardware cannot run.
DTYPE = 'auto' if env['supports_bf16'] else 'float16'
print(f"\nuse dtype={DTYPE!r}")

vram = env['gpu_total_gb']
print('\nwhat fits, roughly:')
for (size, quant), need in sorted(APPROX_VRAM_GB.items()):
    mark = 'yes' if fits(size, quant, vram) else 'no '
    print(f'  {size:>4} {quant:>5}  ~{need:>4.1f} GB   {mark}')

## 3 · Keep the checkpoints somewhere that survives a disconnect

Free runtimes idle out after about 90 minutes and cap around 12 hours. Every cell
writes its own JSON the moment it finishes and the sweep **skips what is already
there**, so re-running section 5 after a disconnect resumes. That only helps if the
files outlive the runtime — mount Drive.

Skip this cell to keep checkpoints in the ephemeral runtime instead.

In [ ]:
OUT = '/content/agent_benchmark'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/onset-hfo/agent_benchmark'
except Exception as exc:
    print(f'not on Colab or Drive declined ({exc.__class__.__name__}); '
          f'checkpoints will not survive a disconnect')
print('checkpoints ->', OUT)

## 4 · Choose the matrix

Start with **one** cell and confirm the loop runs before committing a GPU hour.
`expand_matrix` puts S0 in once per subject rather than once per precision: the fixed
pipeline runs no model, so three identical rows would imply a comparison nobody made.

In [ ]:
from onset_agent.benchmark import Cell, expand_matrix

MODELS = ['Qwen/Qwen2.5-7B-Instruct']   # add 4B / 14B once the loop is proven
SUBJECT = 'sub-pt01'

smoke = [Cell(model=MODELS[0], quantization='4bit', rung='S2', subject=SUBJECT)]
full  = expand_matrix(MODELS, ['4bit', '8bit'], ['S0','S1','S2','S3'], [SUBJECT])

CELLS = smoke          # <- switch to `full` after the smoke test passes
print(f'{len(CELLS)} cell(s)')
for c in CELLS:
    print(' ', c.cell_id)

## 5 · Run it

`backend_factory` is called **once per cell**, so the previous model is released
before the next loads. On a 16 GB card that is the difference between a sweep and an
out-of-memory error.

A cell that fails is recorded with its traceback and the sweep continues — "this
model at this precision could not finish the ladder" is a result, and one model that
will not load should not cost you the other eight.

**Re-run this cell after a disconnect.** It resumes.

In [ ]:
import gc   # torch and DTYPE come from section 2, which must run first
from onset_agent.analysis import AnalysisSession
from onset_agent.backends import make_backend
from onset_agent.benchmark import run_matrix
from onset_hfo.datasets import fetch_slice

def session_factory(subject):
    rec = fetch_slice(subject, 'ictal', '01', t_start=50, t_stop=110)
    return AnalysisSession(rec, verbose=False)

def backend_factory(cell):
    gc.collect(); torch.cuda.empty_cache()   # release the previous model first
    return make_backend('transformers', model=cell.model,
                        quantization=cell.quantization, dtype=DTYPE,
                        revision=None)       # pin a commit here for a citable run

out = run_matrix(CELLS, session_factory=session_factory,
                 backend_factory=backend_factory, out=OUT)

## 6 · Read it

Failed cells keep their row. Check `timing_warnings` before quoting anything from
the `wall_clock_s` column.

In [ ]:
from onset_agent.benchmark import summarise, timing_warnings

frame = summarise(out)
frame.to_csv(f'{out}/summary.csv', index=False)

cols = ['model','quantization','rung','ok','tool_call_validity',
        'unsupported_claim_rate','top5_overlap_vs_s0','n_tool_calls','wall_clock_s']
display(frame[[c for c in cols if c in frame]])

for w in timing_warnings(frame):
    print(f'timing caveat: {w}')

## 7 · The verifier delta

The cost of verification, which is the number nobody reports: true statements the
model verifier removed, against unsupported numbers it let through. It needs a second
model instance, so it is separate from the sweep above — pass `verifier_backend=` to
`run_matrix` to fold it in, at roughly double the cost.

In [ ]:
# from onset_agent.backends import make_backend
# verifier = make_backend('transformers', model=MODELS[0],
#                         quantization='4bit', dtype=DTYPE)
# out = run_matrix(CELLS, session_factory=session_factory,
#                  backend_factory=backend_factory, out=OUT,
#                  verifier_backend=verifier, overwrite=True)

## 8 · Before you quote any of this

* Every cell records its GPU, CUDA version, library versions and git commit. Keep
  `summary.csv` **and** `cells/` — the per-cell JSON holds the drafted report and the
  struck sentences, which is the evidence for the unsupported-claim rate.
* Pin `revision=` to a model commit if the numbers are going in a paper. `main` moves.
* The scripted-planner rows in `docs/ORCHESTRATION.md` are the control. Report both.
* Do not compare `wall_clock_s` across sessions. See section 6.